# MostPop validation sanity — exact full-catalog ranking

Notebook này là bước đầu tiên của giai đoạn baseline. Nó chỉ trả lời một câu hỏi: evaluator có loại strict prior history và tính rank đúng trên toàn bộ frozen training-item catalog hay không?

Phạm vi cố ý hẹp:

- chỉ chạy MostPop;
- chỉ đọc validation target, không đọc test target để tránh test peeking;
- score là training item degree;
- tie-break cố định bằng item_idx tăng dần;
- candidate là toàn bộ training item trừ mapped positive có timestamp nhỏ hơn target;
- mỗi target row có một relevant item, nên Recall@20 là hit-rate theo row;
- xuất một JSON, một Markdown và một hình để Codex kiểm tra.

Kết quả này là sanity baseline, không phải bằng chứng sampler hay GNN tốt hơn.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Không ở Colab; dùng path local nếu bạn đã mount dữ liệu tương đương.')

DRIVE_ROOT = Path('/content/drive/MyDrive/Phase2_Amazon_Audit')
GRAPH_DIR = DRIVE_ROOT / 'g2c_baby_p4'
MANIFEST_PATH = GRAPH_DIR / 'baby_p4_g2c_manifest.json'
OUTPUT_DIR = DRIVE_ROOT / 'mostpop_validation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Thiếu manifest: {MANIFEST_PATH}')

print('Manifest:', MANIFEST_PATH)
print('Output:', OUTPUT_DIR)


## Protocol được freeze trước khi chạy

Notebook không tuning tham số. MostPop chỉ dùng degree trên training graph. Với mỗi validation target, rank chính xác được tính bằng global popularity rank trừ số prior-positive đứng trước target. Cách này tương đương xếp hạng toàn catalog sau khi mask history nhưng không phải tạo ma trận user × item.

Nếu bất kỳ checksum, row count, candidate count hoặc target-history invariant nào sai, notebook dừng ngay.


In [ ]:
from array import array
from collections import Counter, defaultdict
from itertools import groupby
from math import log2
import csv
import gzip
import hashlib
import json
import resource
import time


K = 20


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_artifact(entry, fallback_name):
    recorded = Path(entry['path'])
    fallback = GRAPH_DIR / fallback_name
    if recorded.exists():
        return recorded
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f"Không tìm thấy artifact: {recorded} hoặc {fallback}")


def popularity_order(item_degrees):
    return sorted(range(len(item_degrees)), key=lambda item: (-item_degrees[item], item))


def ranks_from_order(order):
    ranks = [0] * len(order)
    for position, item in enumerate(order, start=1):
        if item < 0 or item >= len(order) or ranks[item] != 0:
            raise AssertionError('Popularity order không phải permutation')
        ranks[item] = position
    if any(rank == 0 for rank in ranks):
        raise AssertionError('Popularity order thiếu item')
    return ranks


def exact_rank_after_history(target_item, prior_items, global_ranks):
    if target_item in prior_items:
        raise AssertionError('Target đã xuất hiện trong strict prior history')
    target_rank = global_ranks[target_item]
    return target_rank - sum(global_ranks[item] < target_rank for item in prior_items)


def top_k_unseen(order, prior_items, k):
    result = []
    for item in order:
        if item not in prior_items:
            result.append(item)
            if len(result) == k:
                return result
    return result


def row_metrics(ranks, k=K):
    ranks = list(ranks)
    if not ranks:
        return {'rows': 0, 'hits_at_k': 0, 'recall_at_k': None, 'ndcg_at_k': None, 'k': k}
    hits = sum(rank <= k for rank in ranks)
    return {
        'rows': len(ranks),
        'hits_at_k': hits,
        'recall_at_k': hits / len(ranks),
        'ndcg_at_k': sum(1.0 / log2(rank + 1) for rank in ranks if rank <= k) / len(ranks),
        'k': k,
    }


def item_cohort(degree):
    if degree >= 397:
        return 'head'
    if degree >= 13:
        return 'body'
    return 'tail'


def user_cohort(degree):
    if degree == 1:
        return 'singleton'
    if degree <= 3:
        return 'repeat_light'
    return 'active'


def grouped_metrics(records, key):
    buckets = defaultdict(list)
    for record in records:
        buckets[record[key]].append(record['rank'])
    total = len(records)
    return {
        label: {
            **row_metrics(ranks),
            'target_share': len(ranks) / total,
        }
        for label, ranks in sorted(buckets.items())
    }


def current_rss_mb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0


In [ ]:
# Oracle nhỏ: shortcut exact-rank phải trùng brute-force sau history mask.
toy_degrees = [2, 5, 5, 1, 2]
toy_order = popularity_order(toy_degrees)
toy_ranks = ranks_from_order(toy_order)
toy_target = 0
toy_prior = {1, 3}
toy_brute_rank = [item for item in toy_order if item not in toy_prior].index(toy_target) + 1
toy_shortcut_rank = exact_rank_after_history(toy_target, toy_prior, toy_ranks)
assert toy_order == [1, 2, 0, 4, 3]
assert toy_brute_rank == toy_shortcut_rank == 2
assert top_k_unseen(toy_order, toy_prior, 2) == [2, 0]
print('Preflight oracle: PASS')


In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
item_count = int(manifest['training_graph']['items'])

train_entry = manifest['artifacts']['train_edges']
validation_entry = manifest['artifacts']['validation_targets']
train_path = resolve_artifact(train_entry, 'baby_p4_train_edges.csv.gz')
validation_path = resolve_artifact(validation_entry, 'baby_p4_validation_targets.csv.gz')

started = time.perf_counter()
print('1/5 Kiểm tra checksum...')
train_sha = sha256_file(train_path)
validation_sha = sha256_file(validation_path)
if train_sha != train_entry['sha256']:
    raise AssertionError('SHA-256 train_edges không khớp manifest')
if validation_sha != validation_entry['sha256']:
    raise AssertionError('SHA-256 validation_targets không khớp manifest')

print('2/5 Đọc validation target...')
validation_targets = []
with gzip.open(validation_path, 'rt', encoding='utf-8', newline='') as handle:
    for row in csv.DictReader(handle):
        validation_targets.append({
            'user_idx': int(row['user_idx']),
            'item_idx': int(row['item_idx']),
            'timestamp_ms': int(row['timestamp_ms']),
            'source_row': int(row['source_row']),
            'candidate_count': int(row['candidate_count']),
        })
if len(validation_targets) != int(validation_entry['rows']):
    raise AssertionError('Validation row count không khớp manifest')

eval_users = {row['user_idx'] for row in validation_targets}
events_by_user = defaultdict(list)
user_train_degree = Counter()
item_degrees = array('I', [0]) * item_count

print('3/5 Quét training graph, tính popularity và lấy history của validation users...')
train_rows = 0
with gzip.open(train_path, 'rt', encoding='utf-8', newline='') as handle:
    for row in csv.DictReader(handle):
        user_idx = int(row['user_idx'])
        item_idx = int(row['item_idx'])
        timestamp_ms = int(row['timestamp_ms'])
        source_row = int(row['source_row'])
        item_degrees[item_idx] += 1
        train_rows += 1
        if user_idx in eval_users:
            user_train_degree[user_idx] += 1
            events_by_user[user_idx].append((timestamp_ms, source_row, item_idx, -1))

if train_rows != int(train_entry['rows']):
    raise AssertionError('Training row count không khớp manifest')
if sum(item_degrees) != train_rows:
    raise AssertionError('Tổng item degree không bằng training edge')

for target_index, target in enumerate(validation_targets):
    events_by_user[target['user_idx']].append((
        target['timestamp_ms'],
        target['source_row'],
        target['item_idx'],
        target_index,
    ))

print('4/5 Dựng deterministic popularity order...')
order = popularity_order(item_degrees)
global_ranks = ranks_from_order(order)

print('5/5 Exact validation ranking với strict-timestamp history...')
results = [None] * len(validation_targets)
unique_recommended = set()
exposure_counts = Counter()
candidate_checks = 0

for user_idx, events in events_by_user.items():
    prior_items = set()
    events.sort(key=lambda event: (event[0], event[1]))
    for _, timestamp_group in groupby(events, key=lambda event: event[0]):
        group = list(timestamp_group)
        target_events = [event for event in group if event[3] >= 0]
        if target_events:
            recommendations = top_k_unseen(order, prior_items, K)
            if len(recommendations) != K:
                raise AssertionError('Candidate catalog nhỏ hơn K')
            recommendation_cohorts = [item_cohort(item_degrees[item]) for item in recommendations]

            for _, _, target_item, target_index in target_events:
                target = validation_targets[target_index]
                expected_candidates = item_count - len(prior_items)
                if expected_candidates != target['candidate_count']:
                    raise AssertionError(
                        f"Candidate count lệch tại source_row={target['source_row']}: "
                        f"{expected_candidates} != {target['candidate_count']}"
                    )
                rank = exact_rank_after_history(target_item, prior_items, global_ranks)
                results[target_index] = {
                    'rank': rank,
                    'item_cohort': item_cohort(item_degrees[target_item]),
                    'user_cohort': user_cohort(user_train_degree[user_idx]),
                }
                candidate_checks += 1
                unique_recommended.update(recommendations)
                exposure_counts.update(recommendation_cohorts)

        for _, _, item_idx, _ in group:
            prior_items.add(item_idx)

if any(record is None for record in results):
    raise AssertionError('Có validation target chưa được evaluate')
if candidate_checks != len(validation_targets):
    raise AssertionError('Số candidate check không bằng validation rows')

ranks = [record['rank'] for record in results]
overall = row_metrics(ranks)
item_metrics = grouped_metrics(results, 'item_cohort')
user_metrics = grouped_metrics(results, 'user_cohort')
exposure_total = sum(exposure_counts.values())
runtime_seconds = time.perf_counter() - started

assertions = {
    'train_sha256_matches_manifest': train_sha == train_entry['sha256'],
    'validation_sha256_matches_manifest': validation_sha == validation_entry['sha256'],
    'train_rows_match_manifest': train_rows == int(train_entry['rows']),
    'validation_rows_match_manifest': len(validation_targets) == int(validation_entry['rows']),
    'item_degree_mass_matches_train_rows': sum(item_degrees) == train_rows,
    'candidate_count_matches_every_target': candidate_checks == len(validation_targets),
    'every_validation_target_evaluated': all(record is not None for record in results),
    'test_targets_not_read': True,
}
if not all(assertions.values()):
    raise AssertionError(assertions)

summary = {
    'status': 'MOSTPOP_VALIDATION_SANITY_EXECUTED',
    'protocol': {
        'split': 'validation_only',
        'score': 'training_item_degree',
        'tie_break': 'item_idx_ascending',
        'candidate_universe': 'all frozen training items minus strict prior mapped positives',
        'timestamp_rule': 'events at the target timestamp are not prior history',
        'target_unit': 'one relevant item per target row',
        'k': K,
    },
    'source': {
        'manifest_path': str(MANIFEST_PATH),
        'manifest_status': manifest['status'],
        'train_edges': {'path': str(train_path), 'rows': train_rows, 'sha256': train_sha},
        'validation_targets': {'path': str(validation_path), 'rows': len(validation_targets), 'sha256': validation_sha},
        'catalog_items': item_count,
        'validation_users': len(eval_users),
    },
    'integrity_assertions': assertions,
    'metrics': overall,
    'target_item_cohorts': item_metrics,
    'user_activity_cohorts': user_metrics,
    'recommendation_exposure': {
        'recommendation_slots': exposure_total,
        'unique_items_at_k': len(unique_recommended),
        'catalog_coverage_at_k': len(unique_recommended) / item_count,
        'item_cohort_share': {
            label: count / exposure_total
            for label, count in sorted(exposure_counts.items())
        },
    },
    'runtime': {
        'wall_seconds': runtime_seconds,
        'process_peak_rss_mb': current_rss_mb(),
    },
    'claim_boundary': (
        'Validation-only MostPop sanity baseline. No test metric, trained recommender, '
        'sampler comparison, GPU profile, or scalability claim.'
    ),
}

print(json.dumps({
    'status': summary['status'],
    'assertions': assertions,
    'metrics': overall,
    'coverage_at_20': summary['recommendation_exposure']['catalog_coverage_at_k'],
    'runtime': summary['runtime'],
}, ensure_ascii=False, indent=2))


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

cohort_order = ['head', 'body', 'tail']
labels = ['Head', 'Body', 'Tail']
ndcg_values = [summary['target_item_cohorts'][name]['ndcg_at_k'] for name in cohort_order]
recall_values = [summary['target_item_cohorts'][name]['recall_at_k'] for name in cohort_order]

fig, ax = plt.subplots(figsize=(10, 5.4))
x = range(len(labels))
width = 0.36
bars_ndcg = ax.bar([value - width / 2 for value in x], ndcg_values, width, label='NDCG@20', color='#3D8DFF')
bars_recall = ax.bar([value + width / 2 for value in x], recall_values, width, label='Recall@20', color='#2B9B75')
ax.set_xticks(list(x), labels)
ax.set_ylabel('Validation metric')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('MostPop validation theo target item cohort')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
for bars in (bars_ndcg, bars_recall):
    ax.bar_label(bars, labels=[f'{bar.get_height():.1%}' for bar in bars], padding=3)
fig.tight_layout()

figure_path = OUTPUT_DIR / '01_mostpop_validation_by_item_cohort.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
import zipfile

summary_path = OUTPUT_DIR / 'mostpop_validation_summary.json'
summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)

def pct(value):
    return 'n/a' if value is None else f'{value:.2%}'

lines = [
    '# MostPop validation sanity',
    '',
    f"- Rows: {summary['metrics']['rows']:,}",
    f"- NDCG@20: {summary['metrics']['ndcg_at_k']:.6f}",
    f"- Recall@20: {summary['metrics']['recall_at_k']:.6f}",
    f"- Catalog Coverage@20: {summary['recommendation_exposure']['catalog_coverage_at_k']:.6f}",
    '',
    '## Theo target item cohort',
    '',
    '| Cohort | Target share | NDCG@20 | Recall@20 |',
    '|---|---:|---:|---:|',
]
for label in ('head', 'body', 'tail'):
    values = summary['target_item_cohorts'][label]
    lines.append(
        f"| {label} | {pct(values['target_share'])} | "
        f"{values['ndcg_at_k']:.6f} | {values['recall_at_k']:.6f} |"
    )
lines.extend([
    '',
    '## Ranh giới claim',
    '',
    summary['claim_boundary'],
    '',
])
report_path = OUTPUT_DIR / 'MOSTPOP_VALIDATION_vn.md'
report_path.write_text('\n'.join(lines), encoding='utf-8')

bundle_path = OUTPUT_DIR / 'mostpop_validation_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (summary_path, report_path, figure_path):
        archive.write(path, arcname=path.name)

print('Đã tạo:', bundle_path)
print('Dung lượng:', bundle_path.stat().st_size, 'bytes')


## Sau khi chạy

1. Cell exact ranking phải in tám assertion đều là true.
2. Không mở hoặc dùng test target ở bước này.
3. Tải nguyên file mostpop_validation_bundle.zip và gửi lại cho Codex.
4. Chỉ sau khi đọc lại bundle mới quyết định đi tiếp BPR-MF.

Nếu runtime quá lâu hoặc RAM tăng bất thường, gửi ảnh cell đang chạy và traceback; không giảm số validation target.


In [ ]:
try:
    from google.colab import files
    files.download(str(bundle_path))
except ImportError:
    print('Bundle nằm tại:', bundle_path)
